In [67]:
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
from sklearn.preprocessing import LabelEncoder
import shutil


In [ ]:
def create_melspectrogram(df, segmented_dir, melspectrogram_dir, all_categories, override=False, n_mels=128):
    segmented_dir = Path(segmented_dir)
    melspectrogram_dir = Path(melspectrogram_dir)
    melspectrogram_list = []
    labels = []

    if not segmented_dir.exists() or not any(segmented_dir.rglob('*.wav')):
        raise FileNotFoundError(
            f"Segmented audio files not found in {segmented_dir}. "
            "Run `clean.ipynb` first to generate segmented audio before extracting features."
        )

    if melspectrogram_dir.exists() and any(melspectrogram_dir.iterdir()):
        if override:
            print('Melspectrogram directory not empty. Cleaning...')
            shutil.rmtree(melspectrogram_dir)
            melspectrogram_dir.mkdir(parents=True)
        else:
            print('Melspectrogram directory not empty and override is False. Returning.')
            return
        
    melspectrogram_dir.mkdir(parents=True, exist_ok=True)

    tqdm._instances.clear()
    for _, f in tqdm(df.iterrows(), total=len(df)):
        for file in segmented_dir.glob(f'{Path(f.fname).stem}_*.wav'):
            y, sr = librosa.load(file, sr=None)
            melspectrogram = librosa.feature.melspectrogram(y=y,
                                                          sr=sr,
                                                          n_mels=n_mels,
                                                          fmax=sr/2)
            melspectrogram_db = librosa.power_to_db(melspectrogram, ref=np.max)
            melspectrogram_list.append(melspectrogram_db)
            labels.append(f.label)

    X = np.array(melspectrogram_list)
    X = np.expand_dims(X, axis=-1)

    label_encoder = LabelEncoder()
    label_encoder.fit(all_categories)
    y = label_encoder.transform(labels)

    np.save(melspectrogram_dir / 'X.npy', X)
    np.save(melspectrogram_dir / 'y.npy', np.array(y))

    with open('../../models/melspectrogram_label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)


In [ ]:
def create_mfcc(df, segmented_dir, mfcc_dir, all_categories, override=False, n_mfcc=13, hop_length=512):
    segmented_dir = Path(segmented_dir)
    mfcc_dir = Path(mfcc_dir)
    mfcc_list = []
    labels = []

    if not segmented_dir.exists() or not any(segmented_dir.rglob('*.wav')):
        raise FileNotFoundError(
            f"Segmented audio files not found in {segmented_dir}. "
            "Run `clean.ipynb` first to generate segmented audio before extracting features."
        )

    if mfcc_dir.exists() and any(mfcc_dir.iterdir()):
        if override:
            print('MFCC directory not empty. Cleaning...')
            shutil.rmtree(mfcc_dir)
            mfcc_dir.mkdir(parents=True)
    else:
        print('MFCC directory isn\'t empty and override is False. Returning.')
        return
    
    mfcc_dir.mkdir(parents=True, exist_ok=True)

    tqdm._instances.clear()
    for _, f in tqdm(df.iterrows(), total=len(df)):
        for file in segmented_dir.glob(f'{Path(f.fname).stem}_*.wav'):
            y, sr = librosa.load(file, sr=None)
            n_fft = min(2048, len(y))
            mfcc = librosa.feature.mfcc(y=y,
                                        sr=sr,
                                        n_mfcc=n_mfcc,
                                        n_fft=n_fft,
                                        hop_length=hop_length)
            mfcc_list.append(mfcc)
            labels.append(f.label)

    X = np.array(mfcc_list)
    X = np.expand_dims(X, axis=-1)

    label_encoder = LabelEncoder()
    label_encoder.fit(all_categories)
    y = label_encoder.transform(labels)

    np.save(mfcc_dir / 'X.npy', X)
    np.save(mfcc_dir / 'y.npy', np.array(y))

    with open('../../models/mfcc_label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)


In [ ]:
def create_mfcc_stats(df, clean_dir, mfcc_stats_dir, override=False, n_mfcc=13, hop_length=512):
    clean_dir = Path(clean_dir)
    mfcc_stats_dir = Path(mfcc_stats_dir)

    if not clean_dir.exists() or not any(clean_dir.rglob('*.wav')):
        raise FileNotFoundError(
            f"Clean audio files not found in {clean_dir}. "
            "Run `clean.ipynb` first and verify `code/kaggle/data/clean/` contains audio before extracting features."
        )

    if mfcc_stats_dir.exists() and any(mfcc_stats_dir.iterdir()):
        if override:
            print('MFCC stats directory not empty. Cleaning...')
            shutil.rmtree(mfcc_stats_dir)
            mfcc_stats_dir.mkdir(parents=True)
        else:
            print('MFCC stats directory not empty and override is False. Returning.')
            return

    mfcc_stats_dir.mkdir(parents=True, exist_ok=True)

    mfcc_stats_list = []
    tqdm._instances.clear()
    for _, f in tqdm(df.iterrows(), total=len(df)):
        y, sr = librosa.load(clean_dir / f.fname, sr=None)
        n_fft = min(2048, len(y))
        mfcc = librosa.feature.mfcc(y=y,
                                    sr=sr,
                                    n_mfcc=n_mfcc,
                                    n_fft=n_fft,
                                    hop_length=hop_length)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)
        mfcc_stats = np.concatenate([mfcc_mean, mfcc_std])
        mfcc_stats_list.append(mfcc_stats)

    X = np.array(mfcc_stats_list)
    y = df['label'].values.tolist()

    np.save(mfcc_stats_dir / 'X.npy', X)
    np.save(mfcc_stats_dir / 'y.npy', np.array(y))


Extract features from files created by `clean.ipynb`


In [ ]:
train_df = pd.read_csv('../../data/meta/train_post_competition.csv')
if not os.path.exists('../../data/meta/train_post_competition.csv'):
    raise FileNotFoundError(
        "Kaggle metadata file not found: code/kaggle/data/meta/train_post_competition.csv. "
        "Download and extract `FSDKaggle2018.meta.zip` into this folder before running feature extraction."
    )

musical_instruments = [
    'Hi-hat', 'Saxophone', 'Trumpet', 'Glockenspiel', 'Cello',
    'Clarinet', 'Oboe', 'Flute', 'Bass_drum', 'Double_bass',
    'Tambourine', 'Cowbell', 'Electric_piano', 'Harmonica',
    'Acoustic_guitar', 'Violin_or_fiddle'
]

train_df = train_df[train_df['label'].isin(musical_instruments)]
train_df = train_df[train_df['fname'].apply(lambda x: os.path.exists(os.path.join('../../data/raw', x)))]
train_df = train_df[train_df['manually_verified'] == 1].reset_index(drop=True)

create_melspectrogram(train_df, '../../data/segmented', '../../features/melspectrograms', musical_instruments,  override=True, n_mels=128)
create_mfcc(train_df, '../../data/segmented', '../../features/mfcc', musical_instruments, override=True, n_mfcc=13, hop_length=512)
create_mfcc_stats(train_df, '../../data/clean', '../../features/mfcc_stats', override=True, n_mfcc=13, hop_length=512)


100%|██████████| 1847/1847 [00:36<00:00, 50.25it/s]


MFCC directory not empty. Cleaning...


100%|██████████| 1847/1847 [00:37<00:00, 49.68it/s]


MFCC stats directory not empty. Cleaning...


100%|██████████| 1847/1847 [00:07<00:00, 256.84it/s]
